In [ ]:
from collections import defaultdict, Counter
import re

label_names = ["O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC","B-MISC","I-MISC"]

def get_entity_set(tokens, tags):
    entities = set()
    start = None
    entity_type = None
    for i, tag in enumerate(tags):
        if isinstance(tag, int):
            tag = label_names[tag]
        if tag.startswith("B-"):
            if start is not None:
                entities.add((start, i, entity_type))
            start = i
            entity_type = tag[2:]
        elif tag.startswith("I-"):
            if start is None:
                start = i
                entity_type = tag[2:]
        else:
            if start is not None:
                entities.add((start, i, entity_type))
                start = None
                entity_type = None
    if start is not None:
        entities.add((start, len(tokens), entity_type))
    return entities

entity_dictionaries = {"PER": set(), "ORG": set(), "LOC": set(), "MISC": set()}
entity_frequency = Counter()

for example in dataset["train"]:
    entities = get_entity_set(example["tokens"], example["ner_tags"])
    for start, end, entity_type in entities:
        text = " ".join(example["tokens"][start:end])
        entity_frequency[text] += 1
        if entity_type in entity_dictionaries:
            entity_dictionaries[entity_type].add(text)

entity_index = defaultdict(list)
for entity_type, entities in entity_dictionaries.items():
    for entity in entities:
        tokens = tuple(entity.split())
        entity_index[tokens[0]].append((tokens, entity_type))

for token in entity_index:
    entity_index[token].sort(key=lambda x: len(x[0]), reverse=True)

ORG_SUFFIXES = {"Inc","Inc.","Ltd","Ltd.","LLC","Corp","Corp.","Corporation","Company"}
PERSON_TITLES = {"Mr","Mr.","Mrs","Mrs.","Ms","Ms.","Dr","Dr.","Prof","Prof.","President","CEO"}
LOCATION_CONTEXT = {"in","from","near","to","towards","toward"}
PERSON_CONTEXT = {"said","met","by"}
ORGANIZATION_CONTEXT = {"company","organization","corporation","firm"}

In [ ]:
def traditional_ner(tokens):
    n = len(tokens)
    predictions = ["O"] * n
    i = 0

    while i < n:
        matched = False
        for entity_tokens, entity_type in entity_index.get(tokens[i], ()):
            length = len(entity_tokens)
            if i + length <= n and tuple(tokens[i:i + length]) == entity_tokens:
                if all(predictions[j] == "O" for j in range(i, i + length)):
                    predictions[i] = "B-" + entity_type
                    for j in range(i + 1, i + length):
                        predictions[j] = "I-" + entity_type
                    matched = True
                    i += length
                    break
        if not matched:
            i += 1

    for i in range(n):
        if i > 0 and predictions[i] == "O":
            if tokens[i - 1] in PERSON_TITLES and re.fullmatch(r"[A-Z][a-z]+", tokens[i]):
                predictions[i] = "B-PER"
        if i > 0 and tokens[i] in ORG_SUFFIXES and predictions[i] == "O":
            predictions[i] = "I-ORG"
            if predictions[i - 1] == "O":
                predictions[i - 1] = "B-ORG"

    for i in range(1, n):
        if predictions[i] != "O" or not re.fullmatch(r"[A-Z][A-Za-z]+", tokens[i]):
            continue
        previous = tokens[i - 1].lower()
        if previous in LOCATION_CONTEXT:
            predictions[i] = "B-LOC"
        elif previous in PERSON_CONTEXT:
            predictions[i] = "B-PER"
        elif previous in ORGANIZATION_CONTEXT:
            predictions[i] = "B-ORG"

    return predictions

In [ ]:
total_gold = 0
total_predicted = 0
total_correct = 0
category_total = Counter()
category_correct = Counter()

for example in dataset["test"]:
    tokens = example["tokens"]
    gold_entities = get_entity_set(tokens, example["ner_tags"])
    predicted_entities = get_entity_set(tokens, traditional_ner(tokens))
    correct_entities = gold_entities & predicted_entities

    total_gold += len(gold_entities)
    total_predicted += len(predicted_entities)
    total_correct += len(correct_entities)

    for start, end, entity_type in gold_entities:
        text = " ".join(tokens[start:end])
        frequency = entity_frequency.get(text, 0)
        category = "COMMON" if frequency >= 5 else "UNCOMMON" if frequency >= 1 else "UNSEEN"
        category_total[category] += 1
        if (start, end, entity_type) in correct_entities:
            category_correct[category] += 1

precision = total_correct / total_predicted if total_predicted else 0
recall = total_correct / total_gold if total_gold else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print(f"Gold entities: {total_gold}")
print(f"Predicted entities: {total_predicted}")
print(f"Correct entities: {total_correct}")
print(f"Precision: {precision:.4%}")
print(f"Recall: {recall:.4%}")
print(f"F1: {f1:.4%}")

for category in ("COMMON", "UNCOMMON", "UNSEEN"):
    total = category_total[category]
    correct = category_correct[category]
    value = correct / total if total else 0
    print(f"{category}: {correct}/{total} ({value:.4%})")

In [ ]:
idx = 4
example = dataset["test"][idx]
tokens = example["tokens"]
gold_entities = get_entity_set(tokens, example["ner_tags"])
traditional_entities = get_entity_set(tokens, traditional_ner(tokens))

print(" ".join(tokens))
print("Gold:", [(" ".join(tokens[s:e]), t) for s, e, t in sorted(gold_entities)])
print("Traditional:", [(" ".join(tokens[s:e]), t) for s, e, t in sorted(traditional_entities)])
print("Uzbekistan frequency:", entity_frequency.get("Uzbekistan", 0))